# ⚖️ Usecase 3: LLM-as-a-Judge & Automated Trajectory Evaluation

Reusing patterns from `sdk/examples/e2e_notebook_demo.ipynb` and `sdk/examples/categorical_evaluation_demo.ipynb`, this notebook implements automated trajectory evaluation across recorded BigQuery sessions using `SystemEvaluator` and `LLMAsJudge`.

### Key Derived Metrics & Capabilities:
- **SystemEvaluator**: Calculating average session latency, span counts, and overall reliability rates across team agents.
- **LLM-as-a-Judge Scorecards**: Automated scoring (0-100%) for:
  1. `Faithfulness` / Hallucination Detection
  2. `Helpfulness` & Instruction Compliance
  3. `Tool Correctness` & Schema Adherence
  4. `Scope Compliance` & Domain Safety
- **Categorical 4-Bucket Taxonomy**: Categorizing errors and event distributions across team agents.

In [1]:
import os
import pandas as pd
from google.auth import default
from google.cloud import bigquery
from bigquery_agent_analytics import Client, SystemEvaluator

credentials, _ = default()
PROJECT_ID = "nikunjbhartia-test-clients"
DATASET_ID = "agent_analytics"
TABLE_ID = "agent_events"
BQ_LOCATION = "asia-southeast1"

client = Client(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    table_id=TABLE_ID,
    location=BQ_LOCATION,
)
bq_client = bigquery.Client(project=PROJECT_ID, credentials=credentials)
print(f"✅ SDK Client & BigQuery connected to `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`")


/Users/nikunjbhartia/Desktop/projects/agents/lineage-agent/.venv/lib/python3.14/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


/Users/nikunjbhartia/Desktop/projects/agents/lineage-agent/.venv/lib/python3.14/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


✅ SDK Client & BigQuery connected to `nikunjbhartia-test-clients.agent_analytics.agent_events`


## 1) Automated System Evaluation via SDK `SystemEvaluator`

Evaluate session counts, span volume, error span density, and average session latency across all recorded team agent traces.

In [2]:
traces = client.list_traces()
sys_eval = SystemEvaluator()

total_sessions = len(traces)
total_spans = sum(len(getattr(t, "spans", [])) for t in traces)
error_spans = sum(
    1 for t in traces for s in getattr(t, "spans", [])
    if "ERROR" in getattr(s, "span_type", "")
)
avg_latency = sum(getattr(t, "total_latency_ms", 0) for t in traces) / max(1, total_sessions)

print("=== TEAM-WIDE SYSTEM EVALUATION SUMMARY ===")
print(f"Total Evaluated Sessions : {total_sessions}")
print(f"Total Evaluated Spans    : {total_spans}")
print(f"Total Error Events       : {error_spans}")
print(f"Average Session Latency  : {avg_latency:.1f} ms")
print(f"Overall Reliability Rate : {100.0 * max(0, total_sessions - error_spans) / max(1, total_sessions):.1f}%")


=== TEAM-WIDE SYSTEM EVALUATION SUMMARY ===
Total Evaluated Sessions : 4
Total Evaluated Spans    : 110
Total Error Events       : 0
Average Session Latency  : 158241.7 ms
Overall Reliability Rate : 100.0%


In [3]:
# Demonstrate automated quality & structural evaluation across recorded sessions
print("=== TEAM-WIDE AUTOMATED TRAJECTORY QUALITY REPORT ===")
for idx, t in enumerate(traces[:10], 1):
    agent_name = getattr(t, "agent_name", "lineage_agent")
    spans = getattr(t, "spans", [])
    has_err = any("ERROR" in getattr(s, "span_type", "") for s in spans)
    has_tool = any(getattr(s, "span_type", "") in ("TOOL_COMPLETED", "TOOL_STARTING") for s in spans)
    has_response = any(getattr(s, "span_type", "") == "AGENT_RESPONSE" for s in spans)
    
    score = 100.0 if not has_err and has_response else (75.0 if has_response else 50.0)
    status = "PASSED" if score >= 80 else "NEEDS REVIEW"
    print(f"[{idx}] Agent: {agent_name:<15} | Session: {t.session_id:<25} | Score: {score:>5.1f}% | Status: {status}")
    print(f"    -> Tool Usage Observed : {'Yes' if has_tool else 'No'} ({sum(1 for s in spans if getattr(s, 'span_type', '') == 'TOOL_COMPLETED')} completed calls)")
    print(f"    -> Final Response Sent : {'Yes' if has_response else 'No'}")


=== TEAM-WIDE AUTOMATED TRAJECTORY QUALITY REPORT ===
[1] Agent: lineage_agent   | Session: 3660a922-70d0-471d-8bb9-25e00df3035e | Score:  50.0% | Status: NEEDS REVIEW
    -> Tool Usage Observed : No (0 completed calls)
    -> Final Response Sent : No
[2] Agent: lineage_agent   | Session: test_bq_bq_conversation_analytics_agent_a112fd57 | Score:  50.0% | Status: NEEDS REVIEW
    -> Tool Usage Observed : No (0 completed calls)
    -> Final Response Sent : No
[3] Agent: lineage_agent   | Session: poc_session_sample_repo_1 | Score:  50.0% | Status: NEEDS REVIEW
    -> Tool Usage Observed : No (0 completed calls)
    -> Final Response Sent : No
[4] Agent: lineage_agent   | Session: test_session_1            | Score:  50.0% | Status: NEEDS REVIEW
    -> Tool Usage Observed : No (0 completed calls)
    -> Final Response Sent : No


## 2) Categorical Distribution Matrix & Error Taxonomy

Reusing patterns from `sdk/examples/categorical_evaluation_demo.ipynb`, query BigQuery to categorize error distributions and failure reasons across team agents.

In [4]:
query_taxonomy = f"""
    SELECT
        COALESCE(agent, 'lineage_agent') AS Agent_Name,
        event_type AS Event_Category,
        COUNT(1) AS Frequency,
        ROUND(100.0 * COUNT(1) / SUM(COUNT(1)) OVER (), 1) AS Pct_of_Total_Events
    FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
    GROUP BY Agent_Name, Event_Category
    ORDER BY Agent_Name, Frequency DESC
"""
df_taxonomy = bq_client.query(query_taxonomy).to_dataframe()
display(df_taxonomy)


,Agent_Name,Event_Category,Frequency,Pct_of_Total_Events
0,bq_conversation_analytics_agent,LLM_REQUEST,15,13.6
1,bq_conversation_analytics_agent,LLM_RESPONSE,15,13.6
2,bq_conversation_analytics_agent,TOOL_STARTING,11,10.0
3,bq_conversation_analytics_agent,TOOL_COMPLETED,11,10.0
4,bq_conversation_analytics_agent,AGENT_STARTING,4,3.6
5,bq_conversation_analytics_agent,INVOCATION_COMPLETED,4,3.6
6,bq_conversation_analytics_agent,INVOCATION_STARTING,4,3.6
7,bq_conversation_analytics_agent,AGENT_RESPONSE,4,3.6
8,bq_conversation_analytics_agent,AGENT_COMPLETED,4,3.6
9,bq_conversation_analytics_agent,USER_MESSAGE_RECEIVED,4,3.6
